# 04 · Demand Forecasting — Global LightGBM Model

This notebook trains the demand-forecasting model and records the reasoning
behind the modeling choices. The reusable logic lives in `src/model.py`
(`train_model`, `rmse`, `mape`, `split_train_val`, feature lists) and
`src/train.py` (`_load_features`); here we exercise it on a single store
(`CA_1`) for fast, interactive iteration. The production run trains on **all**
stores via:

```bash
python src/train.py --horizon-safe
```

and writes `models/lgbm_v1.pkl` plus `outputs/metrics.json`.

In [ ]:
import sys

sys.path.append("..")

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

from src.model import (
    FEATURE_COLS,
    HORIZON_SAFE_FEATURE_COLS,
    TARGET_COL,
    split_train_val,
    train_model,
    rmse,
    mape,
)
from src.train import _load_features

## Why one global model instead of one model per series?

M5 has 30,490 SKU-store series. We deliberately train a **single** LightGBM
model across all of them — the approach taken by the top M5 solutions — rather
than one model per series:

- **Information sharing.** Most series are short, intermittent, and noisy. A
  global model borrows strength across similar items/stores (e.g. weekday and
  SNAP effects learned on dense series transfer to sparse ones).
- **Cold start.** New or thinly-observed SKUs get reasonable forecasts
  immediately from shared structure; a per-series model would have almost
  nothing to fit.
- **Operability.** One artifact to train, version, and serve — versus 30k
  models to retrain and monitor. This is what makes the downstream FastAPI /
  Docker deployment tractable.

The series identity is preserved as **categorical features** (`item_id`,
`store_id`, `dept_id`, `cat_id`, `state_id`), so the single model can still
specialise per series where the data supports it.

## Why LightGBM over the alternatives?

- **Classical (ARIMA / ETS / Prophet):** fit per series, so they don't share
  information and scale poorly to 30k series. They also can't natively absorb
  the rich exogenous features here (price, SNAP, events, calendar).
- **Deep learning (DeepAR / N-BEATS / TFT):** strong but heavier to train,
  tune, and deploy; overkill for tabular features with this signal-to-noise.
- **LightGBM:** fast histogram-based gradient boosting that handles tens of
  millions of rows on a single machine, ingests **categoricals natively**,
  tolerates NaN lag values without imputation, and has a proven M5 track
  record. It is the pragmatic, defensible choice.

## Avoiding forecast leakage: horizon-safe features

This is the subtle, important part. We forecast **28 days ahead** from a fixed
cutoff. A feature is only usable if it's knowable *at the cutoff* for the whole
horizon. Consider `lag_7` (sales 7 days ago) for a horizon day `T+k`: it needs
sales at `T+k-7`. For `k > 7` that's **inside** the horizon — a value we have
not observed yet at forecast time.

So `lag_7`, `lag_14`, and every `shift(1)` rolling feature (`rolling_mean_7`,
`rolling_std_28`, ...) **leak the future** in a genuine multi-step forecast.
Evaluating with them is really "1-step-ahead with oracle lags" and is
optimistic. Only lags `>= 28` and rollings computed on `shift(28)` are
horizon-safe (`HORIZON_SAFE_FEATURE_COLS`); calendar and price features are
known in advance and are fine.

We train on the **horizon-safe** set below. (The leaky set is available as
`FEATURE_COLS` for the optimistic comparison — see `docs/SCDF-18` report.)

## Load features and split temporally

`_load_features` reads the engineered parquet from `data/processed/`,
projecting only the model columns and downcasting to 32-bit. `split_train_val`
performs a strictly **temporal** split — the last 28 days are held out — which
mirrors the M5 evaluation window and prevents leakage from the future into
training.

In [ ]:
with open("../config/model_config.yaml", "r", encoding="utf-8") as fh:
    cfg = yaml.safe_load(fh)

params = cfg["model"]
training_cfg = cfg["training"]
params

In [ ]:
# Single store for notebook speed; `python src/train.py` uses every store.
df = _load_features(Path("../data/processed"), stores=["CA_1"])
df.shape

In [ ]:
feature_cols = HORIZON_SAFE_FEATURE_COLS  # leakage-free multi-step features

train_df, val_df = split_train_val(df, val_days=training_cfg["val_days"])
X_train, y_train = train_df[feature_cols], train_df[TARGET_COL]
X_val, y_val = val_df[feature_cols], val_df[TARGET_COL]
len(feature_cols), X_train.shape, X_val.shape

## Key hyperparameter choices

All hyperparameters live in `config/model_config.yaml` (never hardcoded), so a
config change re-trains without touching code. The values below were picked by
a small subset hyperparameter sweep (see the SCDF-18 report).

- **`objective: tweedie` (`tweedie_variance_power: 1.2`).** Retail demand is
  intermittent — many zero-sales days plus a right-skewed positive tail.
  Tweedie models exactly that mixture and is the M5-winning objective; plain
  squared-error regression would over-predict on the zero-heavy series.
- **`learning_rate: 0.03` with `n_estimators: 3000` + early stopping.** A small
  step size with many rounds, stopped on validation RMSE, generalises better.
- **`num_leaves: 127`, `min_child_samples: 100`.** Capacity vs. overfitting
  control.
- **`feature_fraction` / `bagging_fraction: 0.8`.** Stochastic regularisation
  that also speeds up training.

In [ ]:
model = train_model(
    X_train,
    y_train,
    params,
    X_val,
    y_val,
    early_stopping_rounds=training_cfg["early_stopping_rounds"],
)
model.best_iteration_

## Evaluate on the held-out window

**RMSE** is the primary, leakage-free metric. **MAPE** is a secondary,
business-friendly figure computed only over non-zero actuals — percentage error
is undefined when the true demand is zero, which is common in M5. (Per-segment
error analysis across the ABC-XYZ cells is the focus of the next sprint.)

In [ ]:
preds = model.predict(X_val)

print(f"Validation RMSE : {rmse(y_val, preds):.4f}")
print(f"Validation MAPE : {mape(y_val, preds):.2f}% (non-zero actuals)")

## Which features drive the forecast?

A quick split-count importance ranking (full SHAP analysis is a later story).
Among horizon-safe features, the `lag28` rolling means and calendar position
carry the most signal after item identity.

In [ ]:
importance = (
    pd.Series(model.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
    .head(15)
)
importance